# Linear attention lab

A derive-and-implement linear-attention problem opened USAAIO 2026 Round 2. This is that skill at practice scale. You will write standard attention with batching and a causal mask, derive and write linear attention, time both, and write causal linear attention twice: with prefix sums, and as a recurrent network that reads one token at a time.

**How to work through it.** Each coding part gives you a function to write:
its name, what goes in and what must come out. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under
it. The check prints `Part N passed` when your answer is right and raises
an error that says what is wrong when it is not.

## Setup

Run this cell first. It loads the libraries and defines the helpers every part uses.

In [ ]:
import time
import torch
import torch.nn.functional as F

torch.manual_seed(0)


def phi(x):
    """The positive feature map used throughout: elu(x) + 1 > 0."""
    return F.elu(x) + 1.0


def loop_attention(Q, K, V, weight, causal=False):
    """The definition, one query at a time, for 2-D inputs: weight(q, keys) gives one weight per key."""
    outs = []
    for i in range(Q.shape[0]):
        last = i + 1 if causal else K.shape[0]
        w = weight(Q[i], K[:last])
        outs.append((w[:, None] * V[:last]).sum(0) / w.sum())
    return torch.stack(outs)


softmax_weight = lambda q, keys: torch.exp(keys @ q / q.shape[0] ** 0.5)
linear_weight = lambda q, keys: phi(keys) @ phi(q)
print("setup complete")

## Part 1 (coding task): Standard attention, batched and causal

Write `standard_attention(Q, K, V, causal=False)` for inputs of shape `(..., n, d)`, any number of leading batch dimensions:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\Big(\frac{QK^\top}{\sqrt{d}}\Big)V,
$$

with the softmax over the keys (the last axis of the score matrix). When `causal` is `True`, query $i$ may only attend to keys $j \le i$: set the scores above the diagonal to $-\infty$ with `masked_fill` before the softmax, using a boolean mask from `torch.triu`. Use `transpose(-2, -1)` rather than `.T`, so the function works for any number of batch dimensions.

In [ ]:
def standard_attention(Q, K, V, causal=False):
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = torch.randn(3, 7, 5, 16).unbind(0)                # three (7, 5, 16) tensors: batch of 7
out = standard_attention(Q, K, V)
assert out.shape == (7, 5, 16), f"expected (7, 5, 16), got {tuple(out.shape)}"
for causal in (False, True):
    got = standard_attention(Q, K, V, causal=causal)
    for b in (0, 6):
        want = loop_attention(Q[b], K[b], V[b], softmax_weight, causal=causal)
        assert torch.allclose(got[b], want, atol=1e-5), f"causal={causal}: does not match the definition"
assert torch.allclose(standard_attention(Q, K, V, causal=True)[:, 0], V[:, 0], atol=1e-6), "the first query can only see the first value"
print("Part 1 passed")

## Part 2 (coding task): Linear attention by regrouping the product

Replace the softmax weight $\exp(\mathbf{q}_i\cdot\mathbf{k}_j/\sqrt d)$ by $\phi(\mathbf{q}_i)\cdot\phi(\mathbf{k}_j)$ with the positive feature map `phi` from the setup. The output for query $i$ becomes

$$
\mathbf{o}_i = \frac{\sum_j \big(\phi(\mathbf{q}_i)^\top\phi(\mathbf{k}_j)\big)\,\mathbf{v}_j^\top}{\sum_j \phi(\mathbf{q}_i)^\top\phi(\mathbf{k}_j)}
= \frac{\phi(\mathbf{q}_i)^\top S}{\phi(\mathbf{q}_i)^\top \mathbf{z}},
\qquad S = \sum_j \phi(\mathbf{k}_j)\,\mathbf{v}_j^\top,\quad \mathbf{z} = \sum_j \phi(\mathbf{k}_j).
$$

Write `linear_attention(Q, K, V)` for inputs of shape `(..., n, d)` that computes $S$ (shape `(..., d, d_v)`) and $\mathbf{z}$ (shape `(..., d)`) once and reuses them for every query. It must never form an $n \times n$ matrix.

In [ ]:
def linear_attention(Q, K, V):
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = torch.randn(3, 4, 50, 16).unbind(0)
out = linear_attention(Q, K, V)
assert out.shape == (4, 50, 16), f"expected (4, 50, 16), got {tuple(out.shape)}"
for b in range(4):
    want = loop_attention(Q[b], K[b], V[b], linear_weight)
    gap = (out[b] - want).abs().max().item()
    assert gap < 1e-5, f"batch {b}: differs from the definition by {gap:.2e}"
assert (phi(torch.randn(1000)) > 0).all(), "phi must be strictly positive so the denominator cannot cross zero"
print("Part 2 passed")

## Part 3 (coding task): Measure the two costs

Write `best_time(fn, *args, reps=5)`, which runs `fn(*args)` `reps` times and returns the shortest wall-clock time in seconds, measured with `time.perf_counter`. Then time `standard_attention` and `linear_attention` at $d = 64$ for $n$ in `(256, 512, 1024, 2048, 4096)`, store the results in `times["standard"][n]` and `times["linear"][n]`, and print a table in milliseconds together with the growth factor of each column over the last doubling, from 2048 to 4096.

In [ ]:
def best_time(fn, *args, reps=5):
    # TODO
    raise NotImplementedError


d = 64
times = {"standard": {}, "linear": {}}
# TODO

In [ ]:
assert set(times["standard"]) == {256, 512, 1024, 2048, 4096}, "time every n"
assert best_time(time.sleep, 0.01, reps=2) >= 0.01, "best_time should measure the call"
# Only the large end is checked: small timings are dominated by overhead and noise.
assert times["standard"][4096] > 3 * times["linear"][4096], "at n = 4096 standard attention should be several times slower"
growth = {name: t[4096] / t[2048] for name, t in times.items()}
assert growth["standard"] > 2.5, f"standard attention grew only x{growth['standard']:.1f}, expected about x4"
print("Part 3 passed")

## Part 4 (coding task): Causal linear attention is a recurrent network

For a language model, query $i$ may only use keys and values $j \le i$, so $S$ and $\mathbf{z}$ become **prefix** sums $S_i = \sum_{j \le i}\phi(\mathbf{k}_j)\mathbf{v}_j^\top$ and $\mathbf{z}_i = \sum_{j \le i}\phi(\mathbf{k}_j)$. Write it two ways, for 2-D inputs of shape `(n, d)`.

- `causal_linear_attention(Q, K, V)` computes all the prefix sums at once with `torch.cumsum` over the outer products $\phi(\mathbf{k}_j)\mathbf{v}_j^\top$ (shape `(n, d, d_v)`), then every output with `torch.einsum`.
- `recurrent_linear_attention(Q, K, V)` processes the tokens one at a time, as generation does. It keeps a state $(S, \mathbf{z})$, starting at zero, and for each token adds $\phi(\mathbf{k}_i)\mathbf{v}_i^\top$ to $S$ and $\phi(\mathbf{k}_i)$ to $\mathbf{z}$, then computes $\mathbf{o}_i$ from the updated state. Return the stacked outputs.

In [ ]:
def causal_linear_attention(Q, K, V):
    # TODO
    raise NotImplementedError


def recurrent_linear_attention(Q, K, V):
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = torch.randn(3, 40, 8).unbind(0)
want = loop_attention(Q, K, V, linear_weight, causal=True)
fast, step = causal_linear_attention(Q, K, V), recurrent_linear_attention(Q, K, V)
assert torch.allclose(fast, want, atol=1e-5), "the cumsum form does not match the causal definition"
assert torch.allclose(step, want, atol=1e-5), "the recurrent form does not match the causal definition"

V2 = V.clone()
V2[25] += 10.0                              # change the future
after = causal_linear_attention(Q, K, V2)
assert torch.allclose(fast[:25], after[:25], atol=1e-6), "changing position 25 changed an earlier output: the future leaks"
assert not torch.allclose(fast[25:], after[25:], atol=1e-6), "changing position 25 should change the outputs from 25 on"
print("Part 4 passed")

## Solution

### Part 1: Standard attention, batched and causal

In [ ]:
def standard_attention(Q, K, V, causal=False):
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d ** 0.5            # (..., n, n)
    if causal:
        n = scores.shape[-1]
        future = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))
    return torch.softmax(scores, dim=-1) @ V

**Why it works.** The reference loop computes each query's weights $\exp(\mathbf{q}\cdot\mathbf{k}_j/\sqrt d)$ and normalises them, which is the softmax written out. Masking with $-\infty$ gives the future keys weight $e^{-\infty} = 0$, so each row renormalises over its own prefix. The first query can only see itself, so its output is exactly $\mathbf{v}_1$. The score matrix is $n \times n$, which is the quadratic cost the next parts remove.

### Part 2: Linear attention by regrouping the product

In [ ]:
def linear_attention(Q, K, V):
    Qp, Kp = phi(Q), phi(K)
    S = Kp.transpose(-2, -1) @ V              # (..., d, d_v): one pass over the keys
    z = Kp.sum(dim=-2)                        # (..., d)
    numerator = Qp @ S                        # (..., n, d_v)
    denominator = (Qp * z.unsqueeze(-2)).sum(dim=-1, keepdim=True)    # (..., n, 1)
    return numerator / denominator

**Why it works.** Without the softmax, the output is a product of three matrices, $\phi(Q)\,\phi(K)^\top V$, and matrix multiplication is associative. Grouping it as $(\phi(Q)\phi(K)^\top)V$ costs $O(n^2 d)$, while $\phi(Q)\,(\phi(K)^\top V)$ costs $O(nd^2)$, and the two groupings give the same result. The softmax sits between $QK^\top$ and $V$ and blocks the regrouping, which is why it has to be replaced. `phi` must be positive because its outputs act as weights and the denominator sums them: with negative values the sum could reach zero.

### Part 3: Measure the two costs

In [ ]:
def best_time(fn, *args, reps=5):
    best = float("inf")
    for _ in range(reps):
        start = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - start)
    return best


d = 64
times = {"standard": {}, "linear": {}}
print(f"{'n':>6} {'standard':>11} {'linear':>11}")
for n in (256, 512, 1024, 2048, 4096):
    Q, K, V = torch.randn(3, n, d).unbind(0)
    times["standard"][n] = best_time(standard_attention, Q, K, V)
    times["linear"][n] = best_time(linear_attention, Q, K, V)
    print(f"{n:>6} {times['standard'][n] * 1e3:>9.2f}ms {times['linear'][n] * 1e3:>9.2f}ms")
growth = {name: t[4096] / t[2048] for name, t in times.items()}
print(f"2048 -> 4096: standard x{growth['standard']:.1f}, linear x{growth['linear']:.1f}")

**Why it works.** Doubling $n$ multiplies standard attention's time by roughly 4, the $O(n^2 d)$ score matrix, and linear attention's by roughly 2 or less, the $O(nd^2)$ summaries. At small $n$ the picture is muddier: with $d = 64$ the $nd^2$ term is not much smaller than $n^2 d$ until $n$ is well above $d$, and sub-millisecond timings are dominated by Python and memory-allocation overhead. Linear attention is not a free win on short sequences, which is why the checks only look at the large end.

### Part 4: Causal linear attention is a recurrent network

In [ ]:
def causal_linear_attention(Q, K, V):
    Qp, Kp = phi(Q), phi(K)
    S = torch.cumsum(Kp[:, :, None] * V[:, None, :], dim=0)    # (n, d, d_v): S_i for every i
    z = torch.cumsum(Kp, dim=0)                                # (n, d)
    numerator = torch.einsum("nd,ndv->nv", Qp, S)
    denominator = (Qp * z).sum(dim=-1, keepdim=True)
    return numerator / denominator


def recurrent_linear_attention(Q, K, V):
    Qp, Kp = phi(Q), phi(K)
    S = torch.zeros(Q.shape[1], V.shape[1])                    # the whole state: (d, d_v) ...
    z = torch.zeros(Q.shape[1])                                # ... and (d,)
    outs = []
    for i in range(Q.shape[0]):
        S = S + torch.outer(Kp[i], V[i])
        z = z + Kp[i]
        outs.append((Qp[i] @ S) / (Qp[i] @ z))
    return torch.stack(outs)

**Why it works.** The recurrent form makes the point concrete: the state is a $d \times d_v$ matrix and a $d$-vector, and each new token costs one outer product to update it and one product to read it, $O(d^2)$ work and memory however long the text already is. A standard transformer's key-value cache instead grows with every token, and each step attends over all of it. This is the sense in which causal linear attention is a recurrent network. The price is that a fixed-size state cannot store everything an unbounded cache can, so tasks that need exact recall of distant tokens suffer. The leak test is worth reusing in your own models: a global sum in place of a prefix sum has the same shapes and no error, and it lets every position see the future.